In [2]:
import random
import json
from tqdm import tqdm

# 读取查询和候选文档库
query_file_path = '/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/updated_query.json'
with open(query_file_path, 'r', encoding='utf-8') as f:
    queries = json.load(f)

candidate_file_path = '/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/candidate_base.json'
with open(candidate_file_path, 'r', encoding='utf-8') as f:
    candidate_library = json.load(f)

print(f"加载了 {len(queries)} 条查询")

# 随机打乱查询数据并划分为训练集和测试集
random.shuffle(queries)
split_index = int(0.9 * len(queries))
train_queries = queries[:split_index]
test_queries = queries[split_index:]

# 构造三元组数据集的函数
def construct_tuples(query_subset):
    query_positive_negative_tuples = []

    for query in tqdm(query_subset):
        query_id = query['ridx']
        query_text = query['q']
        candidate_scores = query.get('candidate_scores', {})

        # 筛选正例（得分为3）和负例（得分不为3）
        positive_examples = []
        negative_examples = []

        for candidate_id, score in candidate_scores.items():
            if candidate_id in candidate_library:
                candidate_data = candidate_library[candidate_id]
                document_text = candidate_data.get('ajjbqk', '')

                sample = {
                    'query_id': query_id,
                    'query': query_text,
                    'document_id': candidate_id,
                    'document': document_text,
                    'score': score
                }

                if score == 3:
                    positive_examples.append(sample)
                else:
                    negative_examples.append(sample)

        # 随机采样与正例数量相同的负例并创建三元组
        num_positive = len(positive_examples)
        if num_positive > 0:
            sampled_negatives = random.sample(negative_examples, min(num_positive, len(negative_examples)))
            for positive, negative in zip(positive_examples, sampled_negatives):
                query_positive_negative_tuples.append({
                    'query': query_text,
                    'positive': positive,
                    'negative': negative
                })

    return query_positive_negative_tuples

# 构造训练集的三元组数据
train_tuples = construct_tuples(train_queries)

# 保存训练集三元组和测试集查询到文件
train_output_file_path = '/home/u12321044/share/liang_52/self-correct-retriever/rebuttal/lecard_data/query_positive_negative_tuples.json'
test_output_file_path = '/home/u12321044/share/liang_52/self-correct-retriever/rebuttal/lecard_data/test_queries.json'

with open(train_output_file_path, 'w', encoding='utf-8') as f:
    for tuple_dict in train_tuples:
        f.write(json.dumps(tuple_dict, ensure_ascii=False) + '\n')

with open(test_output_file_path, 'w', encoding='utf-8') as f:
    json.dump(test_queries, f, ensure_ascii=False, indent=4)

print(f"生成了 {len(train_tuples)} 个训练集三元组，并保存了 {len(test_queries)} 条测试集查询")


加载了 107 条查询


100%|██████████| 96/96 [00:00<00:00, 25135.98it/s]


生成了 588 个训练集三元组，并保存了 11 条测试集查询


In [4]:
from tqdm import tqdm

results = []
query_file_path = '/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/updated_query.json'
with open(query_file_path, 'r', encoding='utf-8') as f:
    queries = json.load(f) 
for query in tqdm(queries):
    query_id = query['ridx']
    candidate_scores = query.get('candidate_scores', {})

    for candidate_id, score in candidate_scores.items():
        results.append(f"{query_id} 0 {candidate_id} {score}")

# 将结果写入文件
with open('/home/u12321044/share/liang_52/self-correct-retriever/rebuttal/lecard_data/test_trec.txt', 'w') as f:
    for line in results:
        f.write(line + '\n')


100%|██████████| 107/107 [00:00<00:00, 67823.87it/s]


In [15]:
def calculate_f1(precision_at_5, recall_at_5):
    # 计算 F1 score
    if precision_at_5 + recall_at_5 == 0:
        return 0.0
    return 2 * (precision_at_5 * recall_at_5) / (precision_at_5 + recall_at_5)
print(calculate_f1(93.64,19.02))

31.61783774187822


In [1]:
import random
import json
import re
import cn2an
from tqdm import tqdm

# 读取查询和候选文档库
query_file_path = '/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/updated_query.json'
with open(query_file_path, 'r', encoding='utf-8') as f:
    queries = json.load(f)

candidate_file_path = '/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/candidate_base.json'
with open(candidate_file_path, 'r', encoding='utf-8') as f:
    candidate_library = json.load(f)

print(f"加载了 {len(queries)} 条查询")

# 随机打乱查询数据并划分为训练集和测试集
random.shuffle(queries)
split_index = int(0.9 * len(queries))
train_queries = queries[:split_index]
test_queries = queries[split_index:]

# 记录层级
def hera(text):
    # 定义正则表达式来匹配法条信息
    pattern = r"根据法条第(.+?)条"

    # 定义划分范围
    ranges = [
        (102, 113), (114, 139), (140, 231), (232, 262), (263, 276),
        (277, 367), (368, 381), (382, 396), (397, 419), (420, 451)
    ]
    section_split = {"法条102-113":"危害国家安全罪.json","法条114-139":"危害公共安全罪.json","法条140-231":"破坏社会主义市场经济秩序罪.json",
                    "法条232-262":"侵犯公民人身权利、民主权力罪.json","法条263-276":"侵犯财产罪.json","法条277-367":"妨害社会管理秩序罪.json",
                    "法条368-381":"危害国防利益罪.json","法条382-396":"贪污贿赂罪.json","法条397-419":"渎职罪.json",
                    "法条420-451":"军人违反职责罪.json"}
    # 初始化字典，用于保存每个范围的文本内容
    text_by_range = {range_: [] for range_ in ranges}

    # 使用正则表达式找到所有法条的编号，并按照范围分类保存文本
    match = re.search(pattern,  text.replace("十零","十"))
    if match:
        try:
            article_number = cn2an.cn2an(match.group(1))
        except:
            return ''
        for start, end in ranges:
            if start <= article_number <= end:
                return section_split["法条"+str(start)+"-"+str(end)]

with open("/home/u12321044/share/liang_52/art_case/data/LeCaRD-main/data/artbase/article.json","r") as f:
    art_dic = json.load(f)

# 抽取法条    
def extract_legal_articles(text):
    articles_list = []
    pattern = r'(《[^》]+》)([^《]*?条(?:、[^《]*?条)*)'
    matches = re.findall(pattern, text)
    for law_name, articles_text in matches:
        articles = re.split(r'[、，]', articles_text)

        for article in articles:
            article = article.strip()
            if article and re.match(r'^第[^条]*条$', article):
                full_article = f"{law_name}{article}"
                if full_article in art_dic.keys():
                    articles_list.append(full_article)

    return articles_list


# 构造三元组数据集的函数
def construct_tuples(query_subset):
    query_positive_negative_tuples = []
    i = 0
    for query in tqdm(query_subset):
        query_id = query['ridx']
        query_text = query['q']
        candidate_scores = query.get('candidate_scores', {})

        # 筛选正例（得分为3）和负例（得分不为3）
        positive_examples = []
        negative_examples = []

        for candidate_id, score in candidate_scores.items():
            if candidate_id in candidate_library:
                candidate_data = candidate_library[candidate_id]
                document_text = candidate_data.get('ajjbqk', '')
                byrw = candidate_data.get('cpfxgc', '')
                node_id = [i,i+3026,3026*2+i]
                i += 1
                sample = {
                    'query_id': query_id,
                    'query': query_text,
                    'document_id': candidate_id,
                    'document': document_text,
                    'article': extract_legal_articles(byrw),
                    'document_path': hera(byrw),
                    'node_id':node_id,
                    'score': score
                }

                if score == 3:
                    positive_examples.append(sample)
                else:
                    negative_examples.append(sample)

        # 随机采样与正例数量相同的负例并创建三元组
        num_positive = len(positive_examples)
        if num_positive > 0:
            sampled_negatives = random.sample(negative_examples, min(num_positive, len(negative_examples)))
            for positive, negative in zip(positive_examples, sampled_negatives):
                query_positive_negative_tuples.append({
                    'query': query_text,
                    'positive': positive,
                    'negative': negative
                })

    return query_positive_negative_tuples

# 构造训练集的三元组数据
train_tuples = construct_tuples(train_queries)
# 保存训练集三元组和测试集查询到文件
train_output_file_path = '/home/u12321044/share/liang_52/self-correct-retriever/rebuttal/lecard_data/UniLR_positive_negative_tuples.json'

with open(train_output_file_path, 'w', encoding='utf-8') as f:
    for tuple_dict in train_tuples:
        f.write(json.dumps(tuple_dict, ensure_ascii=False) + '\n')

加载了 107 条查询


100%|██████████| 96/96 [00:00<00:00, 1790.17it/s]


In [3]:
# 读取原始文件
with open('rebuttal/test_trec.txt', 'r') as f:
    lines = f.readlines()

# 读取真实标签文件
with open('rebuttal/Qwen/Lecard_1000_qwenresults.txt', 'r') as f:
    true_labels = [int(line.split(',')[0].strip().split()[-1]) for line in f.readlines()]

# 结果列表
new_lines = []

# 遍历原始文件内容，并替换最后一列为真实标签
for i, line in enumerate(lines):
    parts = line.strip().split()
    
    # 替换最后一列为真实标签，只替换到真实标签列表的结束
    if i < len(true_labels):
        parts[-1] = str(true_labels[i])
        new_lines.append(' '.join(parts))
    else:
        break
    
    # 将处理后的行添加到新列表

# 将新的内容写入到新文件
with open('rebuttal/Qwen/test_trec.txt', 'w') as f:
    for line in new_lines:
        f.write(line + '\n')


In [10]:
from itertools import groupby
# 读取原始文件
with open('rebuttal/test_trec.txt', 'r') as f:
    lines = f.readlines()

# 读取真实标签文件
with open('rebuttal/Qwen/Lecard_1000_qwenresults.txt', 'r') as f:
    pred_labels = [int(line.split(',')[1].strip().split()[-1]) for line in f.readlines()]

# 结果列表
new_lines = []

# 遍历原始文件内容，并替换最后一列为真实标签
for i, line in enumerate(lines):
    parts = line.strip().split()
    
    # 替换最后一列为真实标签，只替换到真实标签列表的结束
    if i < len(pred_labels):
        parts[-1] = pred_labels[i]
        new_lines.append(parts)
    else:
        break

sorted_data = []
query_groups = {}
for entry in new_lines:
    query_id = entry[0]
    if query_id not in query_groups:
        query_groups[query_id] = []
    query_groups[query_id].append(entry)

# 对每个 query_id 组内的条目按预测标签降序排序
for query_id, group in query_groups.items():
    sorted_group = sorted(group, key=lambda x: -x[-1])  # 排序时，使用预测标签（第四列）
    sorted_data.extend(sorted_group)

# 将排序后的结果写入新文件
with open('rebuttal/Qwen/result.txt', 'w') as f:
    rank = 0
    tmp = 0
    for line in sorted_data:
        query_id = line[0]
        if query_id != tmp:
            rank = 0
        tmp = query_id

        doc_id = line[2]
        predicted_label = line[-1]
        f.write(f"{query_id} Q0 {doc_id} {rank} {predicted_label} LHT\n")
        rank +=1
